## Inspect Converted `data/set` Dataset

This notebook inspects the converted dataset stored in `data/set`.

Expected per-file payload:
- one `.npy` file per battery cell
- filename is a 4-character base62 content hash
- array shape is `(cycle, sample, channel)`
- channel mapping: `0 -> voltage_in_V`, `1 -> current_in_A`
- sampling is expected to be at `dt = 1s`

In [ ]:
from pathlib import Path

import numpy as np
import plotly.graph_objects as go

datapath = Path(".")
files = sorted(p for p in datapath.glob("*.npy"))

print(f"datapath: {datapath.resolve()}")
print(f"number of .npy files: {len(files)}")
print("first 10 files:", [p.name for p in files[:10]])

In [ ]:
assert files, "No .npy files found here"

example_path = files[0]
example = np.load(example_path, allow_pickle=False)

print("example file:", example_path.name)
print("dtype:", example.dtype)
print("shape:", example.shape)
assert example.ndim == 3, "Expected tensor shape (cycle, sample, channel)"
assert example.shape[-1] == 2, "Expected 2 channels: voltage/current"

print("example[0, :5, :]:")
print(example[0, :5, :])

### Dataset Consistency Summary

This section checks tensor dimensionality, cycle/sample sizes, and padding behavior (NaN tail padding).

In [ ]:
from statistics import median

cell_rows = []
all_cycle_lengths = []
all_tail_nan_fraction = []
bad_shapes = []

for path in files:
    arr = np.load(path, allow_pickle=False)

    if arr.ndim != 3 or arr.shape[-1] != 2:
        bad_shapes.append((path.name, arr.shape))
        continue

    cycles, max_samples, _ = arr.shape
    finite_mask = np.isfinite(arr[:, :, 0]) & np.isfinite(arr[:, :, 1])
    valid_lengths = finite_mask.sum(axis=1).astype(int)

    cycle_rows = []
    for valid_len in valid_lengths:
        all_cycle_lengths.append(int(valid_len))
        tail_n = max_samples - int(valid_len)
        all_tail_nan_fraction.append(tail_n / max_samples if max_samples else 0.0)
        cycle_rows.append(int(valid_len))

    cell_rows.append(
        {
            "file": path.name,
            "cycles": cycles,
            "max_samples": max_samples,
            "min_valid_samples": min(cycle_rows) if cycle_rows else 0,
            "median_valid_samples": int(np.median(cycle_rows)) if cycle_rows else 0,
            "max_valid_samples": max(cycle_rows) if cycle_rows else 0,
        }
    )

print(f"files inspected: {len(files)}")
print(f"files with valid shape: {len(cell_rows)}")
print(f"files with bad shape: {len(bad_shapes)}")
if bad_shapes:
    print("bad shape examples:", bad_shapes[:5])

if cell_rows:
    cycle_counts = [row["cycles"] for row in cell_rows]
    max_samples_per_cell = [row["max_samples"] for row in cell_rows]

    print(
        "cycles per cell stats:",
        f"min={min(cycle_counts)}, median={median(cycle_counts):.1f}, max={max(cycle_counts)}",
    )
    print(
        "max sample length per cell stats:",
        f"min={min(max_samples_per_cell)}, median={median(max_samples_per_cell):.1f}, max={max(max_samples_per_cell)}",
    )

if all_cycle_lengths:
    print(
        "valid samples per cycle stats:",
        f"min={min(all_cycle_lengths)}, median={median(all_cycle_lengths):.1f}, max={max(all_cycle_lengths)}",
    )

if all_tail_nan_fraction:
    print(
        "tail padding fraction stats:",
        f"min={min(all_tail_nan_fraction):.4f}, median={median(all_tail_nan_fraction):.4f}, max={max(all_tail_nan_fraction):.4f}",
    )

### Training-Oriented Capacity Planning

The next cells estimate:

1. a practical `max_signal_positions` value from dataset signal-length statistics and the configured Conv1D downsampling stack
2. the maximum available `epoch_samples` per split (`train`/`val`) using the same split seed and validation ratio as training

Manual holdout files are not part of this random split analysis; they should be excluded separately when preparing the processed training set.

In [ ]:
import math
from pathlib import Path

import yaml

# Load training defaults so this notebook stays aligned with training config.
cfg_path = Path("../../configs/default.yaml")
with cfg_path.open("r", encoding="utf-8") as handle:
    cfg = yaml.safe_load(handle)

encoder_cfg = cfg["encoder"]
conv_kernels = encoder_cfg["conv_kernels"]
conv_strides = encoder_cfg["conv_strides"]


def conv1d_out_len(
    length: int,
    kernel: int,
    stride: int,
    padding: int,
    dilation: int = 1,
) -> int:
    return math.floor((length + 2 * padding - dilation * (kernel - 1) - 1) / stride + 1)


def downsampled_len(length: int) -> int:
    out = int(length)
    for kernel, stride in zip(conv_kernels, conv_strides, strict=True):
        out = conv1d_out_len(out, kernel=kernel, stride=stride, padding=kernel // 2)
    return max(out, 0)


if not all_cycle_lengths:
    raise RuntimeError(
        "Run the dataset consistency summary cell first to populate all_cycle_lengths."
    )

signal_lengths = np.asarray(all_cycle_lengths, dtype=np.int64)
down_lengths = np.asarray(
    [downsampled_len(int(v)) for v in signal_lengths], dtype=np.int64
)

q95 = int(np.percentile(down_lengths, 95))
q99 = int(np.percentile(down_lengths, 99))
q999 = int(np.percentile(down_lengths, 99.9))
max_len = int(down_lengths.max())

recommended = int(min(max_len, math.ceil(q999 * 1.1)))
configured_max_signal_positions = encoder_cfg.get("max_signal_positions")

if configured_max_signal_positions is None:
    print(
        "config max_signal_positions: not set in current config; use the recommendation below if you want to cap positional length explicitly."
    )
else:
    print("config max_signal_positions:", configured_max_signal_positions)

print("downsampled lengths stats:")
print(
    {
        "min": int(down_lengths.min()),
        "median": int(np.median(down_lengths)),
        "p95": q95,
        "p99": q99,
        "p99_9": q999,
        "max": max_len,
    }
)
print("recommended max_signal_positions:", recommended)
if (
    configured_max_signal_positions is not None
    and configured_max_signal_positions < max_len
):
    print(
        "WARNING: current max_signal_positions is lower than observed max downsampled length."
    )

In [ ]:
import random


def split_files_training_style(
    input_files: list[Path],
    seed: int,
    val_fraction: float,
) -> tuple[list[Path], list[Path]]:
    ordered = sorted(input_files)
    rng = random.Random(seed)
    rng.shuffle(ordered)

    n_total = len(ordered)
    n_val = max(1, round(n_total * val_fraction)) if val_fraction > 0 else 0
    if n_val >= n_total:
        n_val = n_total - 1

    val_files = ordered[:n_val]
    train_files = ordered[n_val:]
    return train_files, val_files


def valid_cycle_count_for_file(path: Path) -> int:
    arr = np.load(path, allow_pickle=False)
    if arr.ndim != 3 or arr.shape[-1] != 2:
        return 0

    valid_cycles = 0
    min_q = float(cfg["data"]["min_discharge_capacity_ah"])
    dt_seconds = float(cfg["data"]["dt_seconds"])

    for cycle_idx in range(arr.shape[0]):
        cycle = arr[cycle_idx]
        valid_mask = np.isfinite(cycle[:, 0]) & np.isfinite(cycle[:, 1])
        if valid_mask.sum() <= 0:
            continue
        current = cycle[valid_mask, 1]
        discharge_capacity_ah = float(
            np.clip(-current, 0.0, None).sum() * dt_seconds / 3600.0
        )

        if cfg["data"]["drop_cycles_without_discharge"]:
            if discharge_capacity_ah >= min_q:
                valid_cycles += 1
        else:
            valid_cycles += 1

    return valid_cycles


def count_windows(
    file_group: list[Path], cycle_window: int
) -> tuple[int, dict[str, int]]:
    total = 0
    per_file = {}
    for path in file_group:
        n_cycles = valid_cycle_count_for_file(path)
        windows = max(0, n_cycles - cycle_window)
        per_file[path.name] = windows
        total += windows
    return total, per_file


data_cfg = cfg["data"]
train_files, val_files = split_files_training_style(
    files,
    seed=int(data_cfg["split_seed"]),
    val_fraction=float(data_cfg["val_fraction"]),
)

cycle_window = int(data_cfg["cycle_window"])

train_max, train_windows_by_file = count_windows(train_files, cycle_window)
val_max, val_windows_by_file = count_windows(val_files, cycle_window)

print("split battery counts:")
print({"train": len(train_files), "val": len(val_files)})

print("max utilize_epoch_windows by split (full available windows):")
print({"train": train_max, "val": val_max})

print("configured utilize_epoch_windows(train):", data_cfg["utilize_epoch_windows"])
if (
    data_cfg["utilize_epoch_windows"] is not None
    and int(data_cfg["utilize_epoch_windows"]) > train_max
):
    print(
        "NOTE: configured train utilize_epoch_windows is larger than available train windows; sampling will use replacement."
    )

print(
    "configured utilize_val_epoch_windows(val):",
    data_cfg["utilize_val_epoch_windows"],
)
if (
    data_cfg["utilize_val_epoch_windows"] is not None
    and int(data_cfg["utilize_val_epoch_windows"]) > val_max
):
    print(
        "NOTE: configured val utilize_val_epoch_windows is larger than available val windows; sampling will use replacement."
    )

### Variable Target-Length Distribution

Under the new sampling rule, each valid start offset contributes one sample, and its target length is the number of cycles remaining after the fixed `cycle_window` context. The next cell visualizes that remaining-horizon distribution directly for both train and validation splits.

In [ ]:
import random
from pathlib import Path

import numpy as np
import plotly.graph_objects as go
import yaml


def split_files_training_style(
    input_files: list[Path],
    seed: int,
    val_fraction: float,
) -> tuple[list[Path], list[Path]]:
    ordered = sorted(input_files)
    rng = random.Random(seed)
    rng.shuffle(ordered)

    n_total = len(ordered)
    n_val = max(1, round(n_total * val_fraction)) if val_fraction > 0 else 0
    if n_val >= n_total:
        n_val = n_total - 1

    val_files = ordered[:n_val]
    train_files = ordered[n_val:]
    return train_files, val_files


def valid_cycle_count_for_file(path: Path, cfg: dict) -> int:
    arr = np.load(path, allow_pickle=False)
    if arr.ndim != 3 or arr.shape[-1] != 2:
        return 0

    valid_cycles = 0
    min_q = float(cfg["data"]["min_discharge_capacity_ah"])
    dt_seconds = float(cfg["data"]["dt_seconds"])

    for cycle_idx in range(arr.shape[0]):
        cycle = arr[cycle_idx]
        valid_mask = np.isfinite(cycle[:, 0]) & np.isfinite(cycle[:, 1])
        if valid_mask.sum() <= 0:
            continue
        current = cycle[valid_mask, 1]
        discharge_capacity_ah = float(
            np.clip(-current, 0.0, None).sum() * dt_seconds / 3600.0
        )

        if cfg["data"]["drop_cycles_without_discharge"]:
            if discharge_capacity_ah >= min_q:
                valid_cycles += 1
        else:
            valid_cycles += 1

    return valid_cycles


def collect_target_lengths(
    file_group: list[Path], cycle_window: int, cfg: dict
) -> tuple[np.ndarray, dict[str, dict[str, int]]]:
    lengths: list[int] = []
    per_file: dict[str, dict[str, int]] = {}
    for path in file_group:
        n_cycles = valid_cycle_count_for_file(path, cfg)
        n_samples = max(0, n_cycles - cycle_window)
        per_file[path.name] = {
            "usable_cycles": n_cycles,
            "num_samples": n_samples,
            "max_target_len": n_samples,
        }
        if n_samples > 0:
            lengths.extend(range(n_samples, 0, -1))
    return np.asarray(lengths, dtype=np.int64), per_file


def print_target_length_summary(label: str, lengths: np.ndarray) -> None:
    if lengths.size == 0:
        print(f"{label}: no valid samples")
        return

    print(f"{label} target-length summary:")
    print(
        {
            "samples": int(lengths.size),
            "min": int(lengths.min()),
            "median": int(np.median(lengths)),
            "p90": int(np.percentile(lengths, 90)),
            "p99": int(np.percentile(lengths, 99)),
            "max": int(lengths.max()),
        }
    )


def ecdf(lengths: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    xs = np.sort(lengths)
    ys = np.arange(1, xs.size + 1, dtype=np.float64) / xs.size
    return xs, ys


cfg_path = Path("../../configs/default.yaml")
with cfg_path.open("r", encoding="utf-8") as handle:
    cfg = yaml.safe_load(handle)

files = sorted(Path(".").glob("*.npy"))
data_cfg = cfg["data"]
cycle_window = int(data_cfg["cycle_window"])
train_files, val_files = split_files_training_style(
    files,
    seed=int(data_cfg["split_seed"]),
    val_fraction=float(data_cfg["val_fraction"]),
)

train_target_lengths, train_target_meta = collect_target_lengths(
    train_files, cycle_window, cfg
)
val_target_lengths, val_target_meta = collect_target_lengths(
    val_files, cycle_window, cfg
)

print_target_length_summary("train", train_target_lengths)
print_target_length_summary("val", val_target_lengths)

hist_fig = go.Figure()
hist_fig.add_trace(
    go.Histogram(
        x=train_target_lengths,
        name="train",
        opacity=0.7,
        nbinsx=80,
    )
)
hist_fig.add_trace(
    go.Histogram(
        x=val_target_lengths,
        name="val",
        opacity=0.7,
        nbinsx=80,
    )
)
hist_fig.update_layout(
    title="Per-Sample Target Length Distribution Under All-Offset Sampling",
    xaxis_title="target length (remaining future cycles after context window)",
    yaxis_title="sample count",
    barmode="overlay",
    template="plotly_white",
)
hist_fig.show()

cdf_fig = go.Figure()
for label, lengths, color in [
    ("train", train_target_lengths, "#1f77b4"),
    ("val", val_target_lengths, "#ff7f0e"),
]:
    if lengths.size == 0:
        continue
    xs, ys = ecdf(lengths)
    cdf_fig.add_trace(
        go.Scatter(
            x=xs,
            y=ys,
            mode="lines",
            name=label,
            line={"color": color},
        )
    )

cdf_fig.update_layout(
    title="Target-Length ECDF",
    xaxis_title="target length (remaining future cycles after context window)",
    yaxis_title="fraction of samples with target length <= x",
    template="plotly_white",
)
cdf_fig.show()

top_train_files = sorted(
    train_target_meta.items(),
    key=lambda item: item[1]["num_samples"],
    reverse=True,
)[:15]

if top_train_files:
    top_fig = go.Figure(
        data=[
            go.Bar(
                x=[name for name, _ in top_train_files],
                y=[meta["num_samples"] for _, meta in top_train_files],
                marker_color="#2a9d8f",
            )
        ]
    )
    top_fig.update_layout(
        title="Top Training Batteries By Number Of Window Samples",
        xaxis_title="battery file",
        yaxis_title="window samples contributed",
        template="plotly_white",
    )
    top_fig.update_xaxes(tickangle=45)
    top_fig.show()

In [ ]:
if cell_rows:
    fig_cycles = go.Figure(
        data=[go.Histogram(x=[r["cycles"] for r in cell_rows], nbinsx=60)]
    )
    fig_cycles.update_layout(
        title="Cycles Per Cell Distribution",
        xaxis_title="Cycles",
        yaxis_title="Count",
        template="plotly_white",
    )
    fig_cycles.show()

if all_cycle_lengths:
    fig_len = go.Figure(data=[go.Histogram(x=all_cycle_lengths, nbinsx=120)])
    fig_len.update_layout(
        title="Valid Samples Per Cycle Distribution",
        xaxis_title="Valid samples in cycle (dt=1s)",
        yaxis_title="Count",
        template="plotly_white",
    )
    fig_len.show()

### Cell-Level Signal Browser

Interactive browser for voltage/current trajectories by file and cycle index.
The x-axis is interpreted as time in seconds because sampling is uniform at `dt=1s`.

In [ ]:
import ipywidgets as widgets
from IPython.display import clear_output, display

tensor_cache = {}


def load_tensor(file_index: int) -> np.ndarray:
    if file_index not in tensor_cache:
        tensor_cache[file_index] = np.load(files[file_index], allow_pickle=False)
    return tensor_cache[file_index]


def get_cycle_valid_length(cycle_array: np.ndarray) -> int:
    finite = np.isfinite(cycle_array[:, 0]) & np.isfinite(cycle_array[:, 1])
    return int(finite.sum())


def make_cycle_figure(arr: np.ndarray, cycle_index: int, file_name: str):
    cycle = arr[cycle_index]
    n_valid = get_cycle_valid_length(cycle)
    if n_valid <= 0:
        n_valid = len(cycle)

    cycle_valid = cycle[:n_valid]
    t = np.arange(n_valid, dtype=np.int32)

    voltage = cycle_valid[:, 0]
    current = cycle_valid[:, 1]

    fig = go.Figure()
    fig.add_trace(
        go.Scatter(
            x=t, y=voltage, mode="lines", name="voltage_in_V", line={"color": "#377eb8"}
        )
    )
    fig.add_trace(
        go.Scatter(
            x=t,
            y=current,
            mode="lines",
            name="current_in_A",
            line={"color": "#e41a1c"},
            yaxis="y2",
        )
    )

    fig.update_layout(
        title=f"{file_name} | cycle_index={cycle_index} | valid_samples={n_valid}",
        xaxis_title="time (s)",
        yaxis={"title": "voltage (V)"},
        yaxis2={"title": "current (A)", "overlaying": "y", "side": "right"},
        template="plotly_white",
        height=520,
    )
    return fig


file_slider = widgets.IntSlider(
    value=0, min=0, max=max(0, len(files) - 1), step=1, description="Cell"
)
cycle_slider = widgets.IntSlider(value=0, min=0, max=0, step=1, description="Cycle")
meta_html = widgets.HTML()
output = widgets.Output()


def clamp_cycle_slider(file_index: int):
    arr = load_tensor(file_index)
    n_cycles = arr.shape[0] if arr.ndim == 3 else 1
    cycle_slider.max = max(0, n_cycles - 1)
    cycle_slider.value = min(cycle_slider.value, cycle_slider.max)


def render(*_):
    file_index = file_slider.value
    arr = load_tensor(file_index)
    file_name = files[file_index].name

    if arr.ndim != 3 or arr.shape[-1] != 2:
        meta_html.value = f"<b>{file_name}</b> has invalid shape: {arr.shape}"
        with output:
            clear_output(wait=True)
            print("Invalid tensor shape for this file.")
        return

    clamp_cycle_slider(file_index)
    cidx = cycle_slider.value
    n_valid = get_cycle_valid_length(arr[cidx])
    meta_html.value = (
        f"<b>File:</b> {file_name}"
        f" &nbsp; <b>Shape:</b> {arr.shape}"
        f" &nbsp; <b>Cycle:</b> {cidx + 1}/{arr.shape[0]}"
        f" &nbsp; <b>Valid samples:</b> {n_valid}"
    )

    fig = make_cycle_figure(arr, cidx, file_name)
    with output:
        clear_output(wait=True)
        fig.show()


file_slider.observe(render, names="value")
cycle_slider.observe(render, names="value")
render()

display(widgets.VBox([file_slider, cycle_slider, meta_html, output]))

### Optional Integrity Spot-Check

Validate filename convention (`^[0-9A-Za-z]{4}\.npy$`) and check for exact duplicate tensors.

In [ ]:
import hashlib
import re

name_pattern = re.compile(r"^[0-9A-Za-z]{4}\.npy$")
bad_names = [p.name for p in files if not name_pattern.match(p.name)]
print("bad names:", bad_names[:10], "(total=", len(bad_names), ")")

seen = {}
duplicate_pairs = []
for path in files:
    arr = np.load(path, allow_pickle=False)
    key = hashlib.sha256(arr.tobytes(order="C")).hexdigest()
    if key in seen:
        duplicate_pairs.append((seen[key], path.name))
    else:
        seen[key] = path.name

print(f"exact duplicate tensors found: {len(duplicate_pairs)}")
if duplicate_pairs:
    print("examples:", duplicate_pairs[:10])